## Georgia EI Analysis Using PyEI

In [ ]:
import pandas as pd
import numpy as np
import json
from pyei.two_by_two import TwoByTwoEI
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import dill

import os
os.environ["PYTENSOR_FLAGS"] = "cxx=,mode=FAST_COMPILE,linker=py"

import shutil
shutil.rmtree(os.path.expanduser("~\\AppData\\Local\\PyTensor"), ignore_errors=True)

### Path and DataFrame

In [ ]:
path = '../output/Georgia/seawulf.csv'

In [ ]:
ga_df = pd.read_csv(path)
ga_df

### Isolate Columns Needed + Create Columns Necessary

[Source](https://github.com/mggg/ecological-inference/blob/main/pyei/intro_notebooks/PyEI_overview.ipynb?short_path=f3688a3)

Whether you are using an example data set or your own data, here is what you will need to pass to the various EI methods:

A vector of length `num_precincts` giving the population of each precincts. Depending on your purposes and data, the appropriate measure of population may be CVAP, VAP, etc. Below we name this `precinct_pops`.

For 2 x 2 EI:

- A vector of length `num_precincts` whose entries are each numbers between zero and 1 that give the fraction of the population of each precinct who belong to the demographic group of interest. Below we name this `group_fraction`.
- A vector of length `num_precincts` whose entries are each numbers beteween zero and 1 that give the fraction of voters in each precinct voting for the candidate of interest (or, if we are estimating turnout, who voted at all). Below, we name this `votes_fraction`.
Optionally: name of the candidate of interest, name of the demographic group of interest, and/or names of precincts (for use in PyEI's plotting and reporting).

In [ ]:
demographics = ['white', 'black', 'latino', 'other']
candidates = ['harris', 'trump']
ga_df.columns = ga_df.columns.str.lower()

ga_data = ga_df[['unique_id', 'kamala d. harris', 'donald j. trump', 'total_votes', 
                 'white_population', 'black_population', 'latino_population', 'other_population', 'total_population']]
ga_data = ga_data.rename(columns={'unique_id' : 'precincts', 'kamala d. harris' : 'harris', 'donald j. trump' : 'trump'})
ga_data['total'] = ga_data['harris'] + ga_data['trump']

# drop rows with no vote information
ga_data = ga_data[ga_data['total'] > 0]
ga_data

In [ ]:
def calculate_shares(df):
    result = df[['precincts', 'total']].copy()

    for demographic in demographics:
        result[f'{demographic}_pct'] = (
            df[f'{demographic}_population'] / df['total_population']
        )

    for candidate in candidates:
        result[f'{candidate}_pct'] = (
            df[candidate] / df['total_votes']
        )

    return result

In [ ]:
data = calculate_shares(ga_data)
data.head()

In [ ]:
data.isna().any()

## Fitting Model

In [ ]:
def fit_king_99(demographic, candidate):
    group_fraction = np.array(data[f"{demographic}_pct"])
    votes_fraction = np.array(data[f"{candidate}_pct"])

    precinct_pops = np.array(data["total"])
    precinct_names = data['precincts']

    ei = TwoByTwoEI(model_name="king99", lmbda=0.5)

    # Fit the model
    ei.fit(group_fraction, 
        votes_fraction, 
        precinct_pops, 
        demographic_group_name=demographic, 
        candidate_name=candidate, 
        precinct_names=precinct_names, 
        draws=1200, # optional
        tune=3000, # optional
        target_accept=.99 # optional
    )

    print(ei.summary())

    return ei


In [ ]:
def fit_king_99_modif(demographic, candidate):
    group_fraction = np.array(data[f"{demographic}_pct"])
    votes_fraction = np.array(data[f"{candidate}_pct"])

    precinct_pops = np.array(data["total"])
    precinct_names = data['precincts']

    ei = TwoByTwoEI(model_name="king99_pareto_modification", pareto_scale=15, pareto_shape=2)

    # Fit the model
    ei.fit(group_fraction, 
        votes_fraction, 
        precinct_pops, 
        demographic_group_name=demographic, 
        candidate_name=candidate, 
        precinct_names=precinct_names, 
        draws=1200, # optional
        tune=3000, # optional
        target_accept=.99 # optional
    )

    print(ei.summary())

    return ei


In [ ]:
models = {}

for candidate in candidates:
    models[candidate] = {}
    for demographic in demographics:
        ei = fit_king_99_modif(demographic, candidate)
        models[candidate][demographic] = ei

### Saving Models

In [ ]:
with open('../output/Georgia/models/ei_models.pkl', 'wb') as f:
    dill.dump(models, f)

### Loading Models

In [ ]:
with open('../output/Georiga/models/ei_models.pkl', 'rb') as f:
    models = dill.load(f)

In [ ]:
models['harris']['black'].plot_kde()

In [ ]:
print(models['harris']['other'].summary())

### Format Fitted Model Data

In [ ]:
posterior_means = models['harris']['white'].posterior_mean_voting_prefs
print("white support for harris", posterior_means[0]) # group
print("non-white support for harris ", posterior_means[1]) # complement

In [ ]:
credible_interval_95 = models['harris']['white'].credible_interval_95_mean_voting_prefs
print("white support for harris", credible_interval_95[0]) # group
print("non-white support for harris ", credible_interval_95[1]) # complement

In [ ]:
sampled_voting_prefs = models['harris']['white'].sampled_voting_prefs # ei.sampled_voting_prefs is samples of district-level voter preference: list of length 2
#sampled_voting_prefs[0] samples of district-wide support of specified group for specified candidate
#sampled_voting_prefs[1] samples of district-wide support of (complement of specified group) for specified candidate

print("white support for harris", sampled_voting_prefs[0]) # group
print("non-white support for harris ", sampled_voting_prefs[1]) # complement

In [ ]:

group_samples = sampled_voting_prefs[0]
complement_samples = sampled_voting_prefs[1]

xs = np.linspace(0, 1, 200)
group_density = gaussian_kde(group_samples)(xs)
complement_density = gaussian_kde(complement_samples)(xs)

density_dict = {
    "group": [{"x": float(x), "y": float(y)} for x, y in zip(xs, group_density)],
    "complement": [{"x": float(x), "y": float(y)} for x, y in zip(xs, complement_density)]
}

### Testing Densities

In [ ]:
group_xs = [pt['x'] for pt in density_dict['group']]
group_ys = [pt['y'] for pt in density_dict['group']]
comp_xs  = [pt['x'] for pt in density_dict['complement']]
comp_ys  = [pt['y'] for pt in density_dict['complement']]

plt.figure(figsize=(8, 4))
plt.plot(group_xs, group_ys, label="Group")
plt.plot(comp_xs, comp_ys, label="Complement")
plt.xlabel("Support proportion")
plt.ylabel("Density")
plt.title("Posterior Density of Candidate Support")
plt.legend()
plt.show()

### Function to Generate Data

In [ ]:
def create_json(models, demographics, candidates):
    ei_json = {"state" : "Georgia", "candidates": []}

    for candidate in candidates:
        cand_entry = {
            "id": candidate.lower(),
            "groups": {}
        }

        for demographic in demographics:
            model = models[candidate][demographic]

            # posterior means
            group_mean, complement_mean = model.posterior_mean_voting_prefs
            posterior_mean = {
                "group": group_mean,
                "complement": complement_mean
            }

            # credible intervals
            group_ci_95, complement_ci_95 = model.credible_interval_95_mean_voting_prefs
            credible_interval_95 = {
                "group": group_ci_95.tolist(),
                "complement": complement_ci_95.tolist()
            }

            # densities
            sampled_voting_prefs = model.sampled_voting_prefs            
            group_samples = sampled_voting_prefs[0]
            complement_samples = sampled_voting_prefs[1]

            xs = np.linspace(0, 1, 200)
            group_density = gaussian_kde(group_samples)(xs)
            complement_density = gaussian_kde(complement_samples)(xs)

            density = {
                "group": [{"x": float(x), "y": float(y)} for x, y in zip(xs, group_density)],
                "complement": [{"x": float(x), "y": float(y)} for x, y in zip(xs, complement_density)]
            }

            cand_entry["groups"][demographic] = {
                "posterior_mean": posterior_mean,
                "credible_interval_95": credible_interval_95,
                "density": density
            }

        ei_json["candidates"].append(cand_entry)
    
    output_path = "../output/Georgia/ga_ei_models.json"

    with open(output_path, "w") as f:
        json.dump(ei_json, f, indent=2)            

In [ ]:
create_json(models, demographics, candidates)